In [26]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from typing import List, Optional, Any, Dict
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb
import numpy as np
from pathlib import Path
import uuid
import os

In [2]:
## create simple text file in the root folder
os.makedirs("../data/text_file", exist_ok=True) 

In [3]:
sample_doc = {
    "../data/text_file/dataset_intro.txt":"""
Checkout Redesign A/B Test
Topics
A/B testingHypothesis testingConfidence intervalsSegment analysisExperiment data quality
The scenario
Etsy: Analyze a four-week checkout experiment, deal with messy tracking, and decide whether the redesign should launch.
Etsy's design team rebuilt the checkout page to cut down on steps, and ran a four-week A/B test in September 2025. You're a product analyst on the checkout team. The designers are keen to launch the redesign to everyone, and the head of product has asked you for a launch decision the team can trust.
Your task
Validate the experiment data, measure the redesign's effect on conversion and order value, and make a clear launch recommendation.
Instructions
.1Load both files and check the experiment's integrity before measuring anything. Look for users who appear in both groups, orders that don't belong in the analysis, and whether the split between groups matches the planned 50/50.
.2Decide how to handle each problem you find, apply it consistently, and report how many users and orders you removed.
.3Use the share of assigned users who placed at least one order after being assigned as the primary metric. Also measure average order value and revenue per user.
.4Compare the two groups on the primary metric with a confidence interval and a significance test, and state your assumptions.
.5Break the results down by device and by new vs. returning visitors. Treat these as follow-up checks and account for testing several segments at once.
.6Estimate what launching would mean for monthly revenue, with a range instead of a single number.
.7Write a one-page launch recommendation: launch, don't launch, or change and retest, with the evidence behind it.
"""
}

In [4]:
for filepath, content in sample_doc.items():
    with open(filepath, "w", encoding="utf-8") as path:
        path.write(content)

print("file is successfully created and store in dataset_file✅")

file is successfully created and store in dataset_file✅


In [5]:
## text loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_file/dataset_intro.txt", encoding="utf-8")

## read the content, first load the content from the loader
content = loader.load()
print(content)

[Document(metadata={'source': '../data/text_file/dataset_intro.txt'}, page_content="\nCheckout Redesign A/B Test\nTopics\nA/B testingHypothesis testingConfidence intervalsSegment analysisExperiment data quality\nThe scenario\nEtsy:\xa0Analyze a four-week checkout experiment, deal with messy tracking, and decide whether the redesign should launch.\nEtsy's design team rebuilt the checkout page to cut down on steps, and ran a four-week A/B test in September 2025. You're a product analyst on the checkout team. The designers are keen to launch the redesign to everyone, and the head of product has asked you for a launch decision the team can trust.\nYour task\nValidate the experiment data, measure the redesign's effect on conversion and order value, and make a clear launch recommendation.\nInstructions\n.1Load both files and check the experiment's integrity before measuring anything. Look for users who appear in both groups, orders that don't belong in the analysis, and whether the split bet

## **Alternative: Directory Loader**

In [6]:
from langchain_community.document_loaders import DirectoryLoader
## this load list of files currently available in the root folder 
## load all the files from the directory
dir_loader = DirectoryLoader(
    "../data/text_file",
    glob="**/**.txt", ## match the pattern
    loader_cls=TextLoader, ## it could be any loader (pdf lodaer)
    loader_kwargs={"encoding":"utf-8"},
    show_progress=False
)

docs = dir_loader.load()

docs

[Document(metadata={'source': '../data/text_file/dataset_intro.txt'}, page_content="\nCheckout Redesign A/B Test\nTopics\nA/B testingHypothesis testingConfidence intervalsSegment analysisExperiment data quality\nThe scenario\nEtsy:\xa0Analyze a four-week checkout experiment, deal with messy tracking, and decide whether the redesign should launch.\nEtsy's design team rebuilt the checkout page to cut down on steps, and ran a four-week A/B test in September 2025. You're a product analyst on the checkout team. The designers are keen to launch the redesign to everyone, and the head of product has asked you for a launch decision the team can trust.\nYour task\nValidate the experiment data, measure the redesign's effect on conversion and order value, and make a clear launch recommendation.\nInstructions\n.1Load both files and check the experiment's integrity before measuring anything. Look for users who appear in both groups, orders that don't belong in the analysis, and whether the split bet

## **Data ingestion into Vector db**

In [7]:
 ## read all the pdf from the directory
def read_process_file(files_dir):
    "Read all the files from directory"
    all_documents = []
    path_dir = Path(files_dir)

    ## access all the files
    docs = list(path_dir.glob("**/*.pdf"))

    print(f"length of files in the doc: {len(docs)}")

    ## fetch each file from the docs list
    for pdf_doc in docs:
        ## feretrieve the file name
        print("\nProcessing files:", pdf_doc.name)

        ## load and read the pdf file
        loader = PyPDFLoader(str(pdf_doc))
        documents = loader.load()

        ## add the information about file
        for doc in documents:
            doc.metadata["source"] = pdf_doc.name
            doc.metadata["file_type"] = "pdf"

        all_documents.extend(documents)
        print("Documents are successfully appended into all_document list")
        print(f"\nTotal number of pages:{len(all_documents)}")
    return all_documents

In [8]:
pdf_documents = read_process_file(files_dir="../data/text_file")

length of files in the doc: 3

Processing files: career_mapping.pdf
Documents are successfully appended into all_document list

Total number of pages:9

Processing files: AI Seekho Bhai Guide.pdf
Documents are successfully appended into all_document list

Total number of pages:14

Processing files: AI Engineering Roadmap.pdf
Documents are successfully appended into all_document list

Total number of pages:17


## **Chunking**

In [9]:
## chunk
def split_docs_chunk(documents, chunk_size=10000, overlap=200):
    ## initialize the object 
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,
                                              chunk_overlap = overlap,
                                              length_function = len,
                                              separators = ["\n\n","\n",","," "]
                                              )

    ## split document using splitter
    chunks = splitter.split_documents(documents)
    print(f"Length of documents: {len(documents)}, splitted chunk length is {len(chunks)}")

    ## show contents
    if chunks:
        print("\nExample of chunks")
        print(f"\nContent: {chunks[0].page_content[:200]}")
        print(f"\nMetadata: {chunks[0].metadata}")
    return chunks

In [10]:
chunks = split_docs_chunk(pdf_documents)
chunks

Length of documents: 17, splitted chunk length is 17

Example of chunks

Content: AI CAREER GUIDE · 2026
You're closer to an
AI Engineer
than you think.
Map your current role to the closest AI specialisation. What skills transfer, what to learn,
and where to start.
 10
 role mappin

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-18T21:16:54+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-18T21:16:54+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'career_mapping.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-18T21:16:54+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-18T21:16:54+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'career_mapping.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content="AI CAREER GUIDE · 2026\nYou're closer to an\nAI Engineer\nthan you think.\nMap your current role to the closest AI specialisation. What skills transfer, what to learn,\nand where to start.\n 10\n role mappings\n 6\n AI specialisations\n 40+\n learning resources\nAI seekho bhai, duniya badal rahi hai.\n@theaimonger · youtube.com/@theaimonger\n linkedin.com/in/prasaadk"),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-18T21:16:54+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-18T21:1

## **Embeddings and Vectordb**

In [23]:
## embedding class
class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"): 
        """ 
        Initialize the embedding model

        Args:
            model_name: str
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    ## load model
    def _load_model(self):
        """ 
        Load embedding model from sentence transformer
        """
        try:
            print(f"Name of the embedding model: {self.model_name}")
            self.model = HuggingFaceEmbeddings(model_name= self.model_name)
            print("Model is successfully loaded")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    ## generate embeddings
    def genearate_embeddings(self, docs: List[str])-> np.ndarray:
        """ Generates embeddings from the list of text"""
        if self.model is None:
            raise ValueError("Model is not loaded.")
        
        embeddings = self.model.embed_documents(texts=docs)
        vector_embeddings = np.asarray(embeddings)
        return vector_embeddings



In [24]:
## create object of embeddings manager
embedding_manager = EmbeddingManager()
embedding_manager

Name of the embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4196.87it/s]


Model is successfully loaded


## **Vector db**

In [13]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_vectordb", persistent_dir: str = "../data/vector_db"):
        """ 
        Initialize the vector database

        Args:
            collection_name: name of the chromadb collection
            persistent_dir: directory to the persistent directory of vector store
        """
        self.collection_name = collection_name
        self.persistent_dir = persistent_dir
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        "Initialize the chromadb client"
        try:
            ## create the directory
            os.makedirs(self.persistent_dir, exist_ok=True)
            ## create client
            self.client = chromadb.PersistentClient(path=self.persistent_dir)

           ## get or create collection in the database
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf embeddings for learning RAG."}
            ) 
            print(f"""vector database is initialize: {self.collection_name}
                  \nThe existing collections in database is: {self.collection.count()}""")
        except Exception as e:
            print("Error initialize of Vector database: ", {e})
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        It helps to store the vector embedding of documents into the vectore database (e.g Chromadb, Pinecone etc.).

        Args:
            documents: List of documents
            embedding: vector embedding form of the corresponding documents.
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match with its corresponding embeddings.")

        ## prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        ## id creation
        for i, (docs, embedding) in enumerate(zip(documents, embeddings)):
            # generates Universally Unique Identifier using version4, 
            # convert them into hexadecimal 32 charcters string and
            # slice that to grab only first 8 characters
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # prepare metadata
            metadata = dict(docs.metadata)
            metadata["doc_index"] = i
            metadata['content_length'] = len(docs.page_content)
            metadatas.append(metadata)

            ## document content
            documents_text.append(docs.page_content)

            ## convert all embedding ndarray into standard list and append
            embedding_list.append(embedding.tolist())

        ## add all information into database
        try:
            if self.collection is None:
                raise RuntimeError("Vector database collection is not initialized.")
            
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents into vector store.")
            print(f"Total collection counts: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Adding documents into vector database: {e}") 

In [14]:
## create object of vector database
vector_database = VectorStore()
vector_database

vector database is initialize: pdf_vectordb
                  
The existing collections in database is: 0


In [25]:
## extract page contents for every chunks
all_text = [doc.page_content for doc in chunks]

## generate the embeddings of it
embedding_text = embedding_manager.genearate_embeddings(docs=all_text)

## store in the vector database
vector_database.add_documents(chunks, embeddings=embedding_text)

Successfully added 17 documents into vector store.
Total collection counts: 17


## **RAG Retriever class**

It is a kind of interface that interacts between user query and vector database 

In [36]:
class RAGRetriever:
    "Handles the query retriever from vector database"
    def __init__(self, embedding_manager: EmbeddingManager, vector_database: VectorStore):
        """ 
        Args: 
            embedding manager: embedding model
            vector database: database where stored already vectorized documents 
        """
        ## Initialize the embeddings and vector
        self.embedding_manager = embedding_manager
        self.vector_database = vector_database

    def retriever(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """ 
        Retrieves the relevant documents for users query

        Args:
            query: User's search query
            top_k: Number of top 5 results based on statistical probability value
            score_threshold: threshold is a similarity value, in which below the value the documents will be discarded.
        """

        ## generate the user's query embedding
        query_embedding = self.embedding_manager.genearate_embeddings(docs=[query])[0]

        ## search in the vector database

        try:
            if self.vector_database.collection is None:
                raise RuntimeError("Vector database collection is not initialized.")
            
            results = self.vector_database.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            ## process results
            retrieve_docs = []

            ## retrieve all information
            if results['documents'] and results['documents'][0]:
                documents = (results.get('documents') or [[]])[0]
                metadatas = (results.get('metadatas') or [[]])[0]
                distances = (results.get('distances') or [[]])[0]
                ids = (results.get('ids') or [[]])[0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1-distance

                    ## similarity checks
                    if similarity_score >= score_threshold:
                        retrieve_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'similarity_score': similarity_score,
                            'rank': i+1

                        }
                    )
                print(f"Retrieve {len(retrieve_docs)} documents after filtering")
            else:
                print("No documents found.")
            return retrieve_docs
        except Exception as e:
            print("Error during retrieval ", e)
            return []

In [37]:
retreiver_rag = RAGRetriever(embedding_manager=embedding_manager, vector_database=vector_database)
retreiver_rag

In [38]:
retreiver_rag.retriever("Can you tell me AI Engineering Roadmap?")

Retrieve 5 documents after filtering


[{'id': 'doc_658ea570_1',
  'content': 'Why this guide exists\nWhen someone says "I\'m an AI engineer" today, they could mean one of at least six completely different\njobs. The field moved faster than job titles could keep up.\nThree years ago, "AI engineer" meant someone who trained models — deep in PyTorch, CUDA, and\nresearch papers. Today it increasingly means someone who uses models to build products. A totally different\nskillset. A totally different career path.\nModel Builders\nTrain, fine-tune & evaluate models. Needs ML\ntheory, maths, research skills. Works at labs or\nlarge enterprises with proprietary model needs.\nModel Users\nBuild products on top of existing foundation\nmodels. Needs software engineering & product\nsense. Where the majority of AI jobs are today.\nThe key insight: nobody needs to start from zero. Your current role already gives you a head start on a\nspecific AI specialisation. This guide maps exactly that.\nThe landscape at a glance\nCurrent role\nAI s